# evaluate.ipynb — test metrics, REAL energy, plots (run after train.ipynb)

Loads the operating points + checkpoints saved by `train.ipynb` (same session, or attach its output as input data).
τ was fixed on the validation split, so nothing here is tuned on the test set. Real energy is measured one sample at a time with a
genuine early exit, interleaved across configs and repeated; the cost-model prediction is shown next to it as a validation.

**Read the MAC-vs-energy plot carefully.** On a T4 at batch size 1, a ViT-Tiny block is launch-bound, so energy tracks the *number of
kernels* rather than the number of tokens or MACs. Token merging adds kernels (matching, gather, scatter). If measured energy does
not follow MACs, that is a real result about proxy mismatch, not a bug: report it (and consider adding a batched-inference measurement
for the merge-only case).

## 0. Install & imports

In [ ]:
!pip install -q timm nvidia-ml-py matplotlib


In [ ]:
# ================= CONFIG (identical block in train.ipynb and evaluate.ipynb) =================
import os, json, time, copy, random, hashlib, math, zipfile
from contextlib import nullcontext
import numpy as np

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"   # only used for my offline test, leave as is on Kaggle

ON_KAGGLE = os.path.isdir("/kaggle/working")
# Data lookup order: cached tensors -> cifar-10-binary.tar.gz in DATA_DIR -> train/ + test/ folders in DATA_DIR -> download.
DATA_DIR = os.environ.get("CIFAR_DIR", "/kaggle/input/datasets/ayush1220/cifar10/cifar10" if ON_KAGGLE else "data/cifar-10")
WORKING_DIR = "/tmp/smoke_working" if SMOKE_TEST else ("/kaggle/working" if ON_KAGGLE else "outputs")
CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints")
RESULTS_DIR = os.path.join(WORKING_DIR, "results")
# Never inside WORKING_DIR: on Kaggle a cache there is saved as output and bloated the download from ~1 MB to 170 MB.
CACHE_DIR = "/tmp/cifar_cache"
for _d in [CHECKPOINT_DIR, RESULTS_DIR, CACHE_DIR]:
    os.makedirs(_d, exist_ok=True)

BACKBONE_NAME = "vit_tiny_patch16_224"
IMG_SIZE = 64 if SMOKE_TEST else 224
NUM_CLASSES = 10
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
SEED = 42
USE_AMP = not SMOKE_TEST          # fp16 autocast on GPU. If you ever hit a dtype error, set False.

# ---- search space ----
EXIT_LAYER_POOL = [2, 4, 6, 8, 10]            # ViT-Tiny has 12 blocks (12 = final classifier)
HEAD_ARCH_POOL = [{"arch": "linear"}, {"arch": "mlp", "hidden": 128}, {"arch": "mlp", "hidden": 192}]
MERGE_RATIO_POOL = [0.0, 0.25, 0.5]           # fraction of patch tokens removed at that exit (max ~0.5 by construction)
MAX_EXITS = 2
TAU_GRID = [0.0, 0.02, 0.05, 0.08, 0.12, 0.16, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # shared entropy threshold

# ---- data budget (all slices are disjoint, carved from the 50k train images) ----
SEARCH_TRAIN_PER_CLASS = 600      # 6,000
SEARCH_VAL_PER_CLASS = 150        # 1,500
SEARCH_CAL_PER_CLASS = 50         # 500
SEARCH_EPOCHS = 4
SEARCH_POP = 12
SEARCH_GENS = 5
FINAL_TRAIN_PER_CLASS = 2000      # 20,000  (raise towards 4,500 for more data; test set is separate)
FINAL_CAL_PER_CLASS = 100         # 1,000   (temperature scaling)
FINAL_VAL_PER_CLASS = 200         # 2,000   (threshold / operating-point selection)
FINAL_EPOCHS = 10

EPS = 0.01                        # no-regression tolerance: accuracy >= baseline - EPS
LR = 2e-3
BATCH_SIZE = 128

# ---- measurement ----
N_ENERGY_REPEATS = 5
REAL_ENERGY_SAMPLES = 2000        # test images for the real single-sample early-exit energy pass
PROFILE_TARGET_SECONDS = 10.0     # how long each cost-model micro-benchmark runs
CARBON_INTENSITY_KG_PER_KWH = 0.4 # ASSUMPTION (grid average). Change to your region's figure; state it in the paper.

if SMOKE_TEST:
    SEARCH_TRAIN_PER_CLASS, SEARCH_VAL_PER_CLASS, SEARCH_CAL_PER_CLASS = 8, 4, 3
    SEARCH_EPOCHS, SEARCH_POP, SEARCH_GENS = 1, 3, 2
    FINAL_TRAIN_PER_CLASS, FINAL_CAL_PER_CLASS, FINAL_VAL_PER_CLASS = 20, 5, 6
    FINAL_EPOCHS = 2
    REAL_ENERGY_SAMPLES, N_ENERGY_REPEATS, PROFILE_TARGET_SECONDS = 20, 2, 0.05


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import pandas as pd

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

device = "cuda" if torch.cuda.is_available() else "cpu"
# Must match the pretrained weights. augreg ViT-Tiny expects 0.5/0.5; ImageNet stats here cost about 14 accuracy points.
MODEL_MEAN = torch.tensor((0.5, 0.5, 0.5), device=device).view(1, 3, 1, 1)
MODEL_STD = torch.tensor((0.5, 0.5, 0.5), device=device).view(1, 3, 1, 1)

def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if (USE_AMP and device == "cuda") else nullcontext()

set_seed(SEED)
print("device:", device, "| AMP:", USE_AMP and device == "cuda", "| smoke test:", SMOKE_TEST)

import matplotlib.pyplot as plt


In [ ]:
# ---------------- data: decode once, keep uint8 on the GPU, resize/normalize on the GPU ----------------
# (No DataLoader workers -> removes the CPU decode/resize bottleneck and the worker hangs on Kaggle.)
def find_split_dir(root, split_name):
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.basename(dirpath) == split_name:
            return dirpath
    raise FileNotFoundError(f"No '{split_name}' directory under {root}")

def read_split_to_uint8(split_dir):
    from PIL import Image
    from torchvision import datasets
    ds = datasets.ImageFolder(split_dir)
    n = len(ds)
    X = np.empty((n, 32, 32, 3), dtype=np.uint8)
    Y = np.empty((n,), dtype=np.int64)
    for i, (path, label) in enumerate(ds.samples):
        img = Image.open(path).convert("RGB")
        if img.size != (32, 32):
            img = img.resize((32, 32), Image.BILINEAR)
        X[i] = np.asarray(img); Y[i] = label
        if (i + 1) % 10000 == 0:
            print(f"  decoded {i + 1}/{n}")
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(Y), ds.classes

CIFAR_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz"
CIFAR_MD5 = "c32a1d4ab5d03f1284b67883e8d87530"

def read_cifar_tar(path):
    # Official binary format: each record is 1 label byte + 3072 image bytes. Raw bytes, so no pickle is involved.
    import tarfile
    def part(tf, names):
        raw = b"".join(tf.extractfile(m).read() for m in sorted(names))
        a = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3073)
        return torch.from_numpy(a[:, 1:].reshape(-1, 3, 32, 32).copy()), torch.from_numpy(a[:, 0].astype(np.int64))
    with tarfile.open(path) as tf:
        names = tf.getnames()
        Xtr, Ytr = part(tf, [n for n in names if "data_batch_" in n])
        Xte, Yte = part(tf, [n for n in names if "test_batch" in n])
    assert len(Ytr) == 50000 and len(Yte) == 10000, "unexpected CIFAR-10 archive contents"
    return Xtr, Ytr, Xte, Yte

def download_cifar():
    import urllib.request
    path = os.path.join(CACHE_DIR, "cifar-10-binary.tar.gz")
    print("downloading CIFAR-10 (162 MB):", CIFAR_URL)
    urllib.request.urlretrieve(CIFAR_URL, path)
    md5 = hashlib.md5(open(path, "rb").read()).hexdigest()
    assert md5 == CIFAR_MD5, f"CIFAR-10 checksum mismatch: {md5}"
    return path

def load_cifar_tensors():
    if SMOKE_TEST:
        g = torch.Generator().manual_seed(0)
        def synth(n):
            Y = torch.arange(n) % NUM_CLASSES
            X = (torch.rand(n, 3, 32, 32, generator=g) * 80 + Y.view(-1, 1, 1, 1) * 15).clamp(0, 255).to(torch.uint8)
            return X, Y.long()
        Xtr, Ytr = synth(800); Xte, Yte = synth(300)
        return Xtr, Ytr, Xte, Yte
    cache = os.path.join(CACHE_DIR, "cifar10_uint8.pt")
    if os.path.exists(cache):
        d = torch.load(cache)
        return d["Xtr"], d["Ytr"], d["Xte"], d["Yte"]
    tar = os.path.join(DATA_DIR, "cifar-10-binary.tar.gz")
    if os.path.isfile(tar):
        Xtr, Ytr, Xte, Yte = read_cifar_tar(tar)
    elif os.path.isdir(DATA_DIR):
        train_dir, test_dir = find_split_dir(DATA_DIR, "train"), find_split_dir(DATA_DIR, "test")
        print("train dir:", train_dir, "| test dir:", test_dir)
        Xtr, Ytr, classes = read_split_to_uint8(train_dir)
        Xte, Yte, _ = read_split_to_uint8(test_dir)
        assert list(classes) == CLASS_NAMES, f"class order differs from CLASS_NAMES: {classes}"
    else:
        Xtr, Ytr, Xte, Yte = read_cifar_tar(download_cifar())
    torch.save({"Xtr": Xtr, "Ytr": Ytr, "Xte": Xte, "Yte": Yte}, cache)
    return Xtr, Ytr, Xte, Yte

def prepare_batch(xb_uint8, augment=False):
    x = xb_uint8.float().div_(255.0)
    if augment:
        flip = torch.rand(x.size(0), device=x.device) < 0.5
        x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    return (x - MODEL_MEAN) / MODEL_STD

class GPUData:
    def __init__(self, X, Y):
        self.X, self.Y = X.to(device), Y.to(device)
    def batches(self, idx, batch_size, shuffle=False, augment=False):
        idx_t = torch.as_tensor(idx, dtype=torch.long, device=device)
        if shuffle:
            idx_t = idx_t[torch.randperm(len(idx_t), device=device)]
        for i in range(0, len(idx_t), batch_size):
            b = idx_t[i:i + batch_size]
            yield prepare_batch(self.X[b], augment), self.Y[b]

def stratified_indices(labels_np, per_class, seed, exclude=None):
    exclude = exclude or set()
    rng = np.random.RandomState(seed)
    chosen = []
    for c in range(NUM_CLASSES):
        idxs = np.array([i for i in np.where(labels_np == c)[0] if i not in exclude])
        rng.shuffle(idxs)
        chosen.extend(idxs[:per_class].tolist())
    return chosen


In [ ]:
# ---------------- model: frozen ViT backbone + exit heads + token merging ----------------
def merge_pair_count(Np, ratio):
    # number of patch tokens removed. ratio = fraction of patch tokens removed (capped: bipartite matching can
    # remove at most one token per token of set A, so the practical maximum is ~0.5).
    if ratio <= 0 or Np < 4:
        return 0
    a_len = (Np + 1) // 2
    return int(min(max(1, int(Np * ratio)), a_len - 1))

def merged_patch_count(Np, ratio):
    return Np - merge_pair_count(Np, ratio)

def bipartite_soft_matching_merge(tokens, ratio):
    # Simplified ToMe-style merge: split patch tokens into sets A/B, match every A token to its most similar B token
    # (cosine), merge the r most similar pairs by mean. CLS (index 0) is protected. Done in fp32 for safety.
    B, N, D = tokens.shape
    r = merge_pair_count(N - 1, ratio)
    if r <= 0:
        return tokens
    dtype = tokens.dtype
    t = tokens.float()
    cls_tok, patch = t[:, :1], t[:, 1:]
    a, b = patch[:, 0::2], patch[:, 1::2]
    with torch.no_grad():
        scores = F.normalize(a, dim=-1) @ F.normalize(b, dim=-1).transpose(-1, -2)
        node_max, node_idx = scores.max(dim=-1)
        order = node_max.argsort(dim=-1, descending=True)
        src_idx, unm_idx = order[:, :r], order[:, r:]
        dst_idx = node_idx.gather(-1, src_idx)
    D_ = a.shape[-1]
    unm = a.gather(1, unm_idx.unsqueeze(-1).expand(-1, -1, D_))
    src = a.gather(1, src_idx.unsqueeze(-1).expand(-1, -1, D_))
    b = b.scatter_reduce(1, dst_idx.unsqueeze(-1).expand(-1, -1, D_), src, reduce="mean", include_self=True)
    return torch.cat([cls_tok, unm, b], dim=1).to(dtype)

def embed_tokens(backbone, x):
    x = backbone.patch_embed(x)
    x = backbone._pos_embed(x)
    if hasattr(backbone, "norm_pre"):
        x = backbone.norm_pre(x)
    return x

class OffRampHead(nn.Module):
    def __init__(self, dim, num_classes, arch="linear", hidden=128):
        super().__init__()
        body = nn.Linear(dim, num_classes) if arch == "linear" else \
            nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, num_classes))
        # LayerNorm first: intermediate residual-stream CLS tokens are un-normalised and their scale grows with depth.
        self.net = nn.Sequential(nn.LayerNorm(dim), body)
    def forward(self, x):
        return self.net(x)

def norm_cfg(cfg):
    return {"exit_layers": sorted(int(l) for l in cfg["exit_layers"]),
            "head_arch": {int(k): dict(v) for k, v in cfg["head_arch"].items()},
            "merge_ratio": {int(k): float(v) for k, v in cfg["merge_ratio"].items()}}

def cfg_hash(cfg):
    return hashlib.md5(json.dumps(norm_cfg(cfg), sort_keys=True).encode()).hexdigest()[:8]

def cfg_label(cfg):
    cfg = norm_cfg(cfg)
    if not cfg["exit_layers"]:
        return "full"
    parts = []
    for l in cfg["exit_layers"]:
        h = cfg["head_arch"][l]
        hs = "lin" if h["arch"] == "linear" else f"mlp{h['hidden']}"
        parts.append(f"L{l}:{hs}:m{cfg['merge_ratio'].get(l, 0.0)}")
    return " + ".join(parts)

class EarlyExitViT(nn.Module):
    def __init__(self, backbone, num_classes, exit_layers=None, head_arch=None, merge_ratio=None):
        super().__init__()
        self.backbone = backbone
        self.exit_layers = sorted(exit_layers or [])
        head_arch, merge_ratio = head_arch or {}, merge_ratio or {}
        self.heads = nn.ModuleDict({
            str(l): OffRampHead(backbone.embed_dim, num_classes, **head_arch.get(l, {"arch": "linear"}))
            for l in self.exit_layers})
        self.merge_ratio = {l: merge_ratio.get(l, 0.0) for l in self.exit_layers}
        self.final_head = nn.Linear(backbone.embed_dim, num_classes)
        self.temps = {l: 1.0 for l in self.exit_layers}; self.temps["final"] = 1.0

    def forward(self, x):
        # Batched: every sample runs the full network and we return ALL exit logits. Exit decisions are applied
        # offline (see sweep_points). The frozen backbone runs under no_grad; only heads receive gradients.
        with torch.no_grad():
            tokens = embed_tokens(self.backbone, x)
        all_logits = {}
        for i, block in enumerate(self.backbone.blocks, start=1):
            with torch.no_grad():
                tokens = block(tokens)
            if i in self.exit_layers:
                all_logits[i] = self.heads[str(i)](tokens[:, 0, :]).float()
                if self.merge_ratio.get(i, 0.0) > 0:
                    with torch.no_grad():
                        tokens = bipartite_soft_matching_merge(tokens, self.merge_ratio[i])
        with torch.no_grad():
            feat = self.backbone.norm(tokens)[:, 0, :]
        all_logits["final"] = self.final_head(feat).float()
        return all_logits

    @torch.no_grad()
    def forward_single_early_exit(self, x, tau):
        # x: [1, C, H, W]. TRUE early exit: the loop really stops, so the remaining blocks are never executed.
        tokens = embed_tokens(self.backbone, x)
        for i, block in enumerate(self.backbone.blocks, start=1):
            tokens = block(tokens)
            if i in self.exit_layers:
                logits = self.heads[str(i)](tokens[:, 0, :]).float()
                p = F.softmax(logits / self.temps.get(i, 1.0), dim=-1)
                ent = -(p * torch.log(p.clamp_min(1e-12))).sum() / math.log(p.numel())
                if ent.item() < tau:
                    return logits, i
                if self.merge_ratio.get(i, 0.0) > 0:
                    tokens = bipartite_soft_matching_merge(tokens, self.merge_ratio[i])
        feat = self.backbone.norm(tokens)[:, 0, :]
        return self.final_head(feat).float(), "final"

def save_ckpt(model, cfg, temps, path):
    torch.save({"config": norm_cfg(cfg), "final_head": model.final_head.state_dict(),
                "heads": {str(l): model.heads[str(l)].state_dict() for l in model.exit_layers},
                "temps": {str(k): float(v) for k, v in temps.items()}}, path)

def parse_temps(d):
    return {(int(k) if str(k).isdigit() else k): float(v) for k, v in d.items()}

def load_ckpt(path, backbone):
    ck = torch.load(path, map_location=device, weights_only=False)
    cfg = norm_cfg(ck["config"])
    model = EarlyExitViT(backbone, NUM_CLASSES, **cfg).to(device)
    model.final_head.load_state_dict(ck["final_head"])
    for l, sd in ck["heads"].items():
        model.heads[str(l)].load_state_dict(sd)
    model.temps = parse_temps(ck["temps"])
    model.eval()
    return model, cfg, model.temps

@torch.no_grad()
def collect_logits(model, data, idx, batch_size=256):
    model.eval()
    outs, labels = {}, []
    for x, y in data.batches(idx, batch_size):
        with amp_ctx():
            lg = model(x)
        for k, v in lg.items():
            outs.setdefault(k, []).append(v.float().cpu())
        labels.append(y.cpu())
    return {k: torch.cat(v) for k, v in outs.items()}, torch.cat(labels)

def make_backbone():
    kw = {"img_size": IMG_SIZE} if SMOKE_TEST else {}
    bb = timm.create_model(BACKBONE_NAME, pretrained=not SMOKE_TEST, num_classes=0, **kw).to(device)
    cfg_mean, cfg_std = bb.pretrained_cfg["mean"], bb.pretrained_cfg["std"]
    assert np.allclose(cfg_mean, MODEL_MEAN.flatten().tolist()) and np.allclose(cfg_std, MODEL_STD.flatten().tolist()), \
        f"input normalisation does not match the weights: expected mean={cfg_mean} std={cfg_std}"
    for p in bb.parameters():
        p.requires_grad = False
    bb.eval()
    geom = dict(num_patches=bb.patch_embed.num_patches, dim=bb.embed_dim, blocks=len(bb.blocks))
    return bb, geom


In [ ]:
# ---------------- cost models: measured energy (NVML) vs analytic MACs vs token-layers ----------------
class EnergyMeter:
    # Reads the cumulative GPU energy counter via NVML (millijoules) on GPU 0 only -> real measured joules.
    # Falls back to wall-clock seconds if NVML is unavailable (unit then reads "s"; fine for relative comparisons).
    def __init__(self, gpu_index=0):
        self.ok, self.unit = False, "s"
        if device == "cuda":
            try:
                import pynvml
                pynvml.nvmlInit()
                self.nv = pynvml
                self.h = pynvml.nvmlDeviceGetHandleByIndex(gpu_index)
                pynvml.nvmlDeviceGetTotalEnergyConsumption(self.h)
                self.ok, self.unit = True, "J"
            except Exception as e:
                print("NVML energy counter unavailable, falling back to wall-clock seconds:", e)
        print("EnergyMeter unit:", self.unit)
    def read(self):
        if device == "cuda":
            torch.cuda.synchronize()
        return self.nv.nvmlDeviceGetTotalEnergyConsumption(self.h) / 1000.0 if self.ok else time.perf_counter()

def time_energy(fn, meter, target_seconds, repeats, warmup=10):
    for _ in range(warmup):
        fn()
    meter.read()
    t0 = time.perf_counter()
    for _ in range(20):
        fn()
    meter.read()
    t_call = (time.perf_counter() - t0) / 20
    reps = max(30, int(target_seconds / max(t_call, 1e-6)))
    vals = []
    for _ in range(repeats):
        e0 = meter.read()
        for _ in range(reps):
            fn()
        e1 = meter.read()
        vals.append((e1 - e0) / reps)
    return float(np.median(vals))

class MACCostModel:
    # Analytic multiply-accumulates (includes the attention matmuls, which fvcore skips for fused attention).
    def __init__(self, geom, num_classes=NUM_CLASSES):
        self.D, self.Np, self.C = geom["dim"], geom["num_patches"], num_classes
    def embed(self): return self.Np * (16 * 16 * 3) * self.D
    def block(self, N): return 12 * N * self.D * self.D + 2 * N * N * self.D
    def head(self, arch):
        if arch["arch"] == "linear": return self.D * self.C
        h = arch["hidden"]; return self.D * h + h * self.C
    def merge(self, N, r):
        Np = N - 1; a = (Np + 1) // 2; b = Np // 2
        return a * b * self.D
    def final(self): return self.D * self.C

class TokenCostModel:
    # "token-layers": number of tokens pushed through transformer blocks (the transformer analogue of token cost)
    def embed(self): return 0.0
    def block(self, N): return float(N)
    def head(self, arch): return 0.0
    def merge(self, N, r): return 0.0
    def final(self): return 0.0

class MeasuredCostModel:
    def __init__(self, table):
        self.t = table
        self.unit = table.get("unit", "J")
        self.bk = sorted(int(k) for k in table["block"])
    def embed(self): return self.t["embed"]
    def block(self, N):
        return float(np.interp(N, self.bk, [self.t["block"][str(k)] for k in self.bk]))
    def head(self, arch): return self.t["head"][json.dumps(arch, sort_keys=True)]
    def merge(self, N, r):
        key = f"{N}_{r}"
        if key in self.t["merge"]:
            return self.t["merge"][key]
        cands = [(abs(int(k.split("_")[0]) - N), v) for k, v in self.t["merge"].items() if float(k.split("_")[1]) == r]
        return min(cands)[1] if cands else 0.0
    def final(self): return self.t["final"]

def path_costs(cfg, cm, geom):
    # cost (in the cost model's unit) of a sample that leaves at each exit, or runs to the end ('final').
    cfg = norm_cfg(cfg)
    exits = set(cfg["exit_layers"])
    N = 1 + geom["num_patches"]
    cost, costs = cm.embed(), {}
    for layer in range(1, geom["blocks"] + 1):
        cost += cm.block(N)
        if layer in exits:
            cost += cm.head(cfg["head_arch"][layer])
            costs[layer] = cost
            r = cfg["merge_ratio"].get(layer, 0.0)
            if r > 0:
                cost += cm.merge(N, r)
                N = 1 + merged_patch_count(N - 1, r)
    costs["final"] = cost + cm.final()
    return costs


In [ ]:
# ---------------- calibration, exit decisions, Pareto front, knee, operating points ----------------
def fit_temperature_grid(logits, labels):
    # robust: grid search over T minimising NLL (replaces the LBFGS version that collapsed to its lower clamp)
    grid = torch.logspace(-1.3, 1.0, 200)
    nll = torch.stack([F.cross_entropy(logits / t, labels) for t in grid])
    return float(grid[nll.argmin()].item())

def fit_all_temps(logits, labels):
    return {k: fit_temperature_grid(v, labels) for k, v in logits.items()}

def expected_calibration_error(conf, correct, n_bins=15):
    conf, correct = np.asarray(conf), np.asarray(correct).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def normalized_entropy(logits, T):
    p = F.softmax(logits / T, dim=-1)
    return -(p * torch.log(p.clamp_min(1e-12))).sum(-1) / math.log(logits.shape[1])

def sweep_points(logits, labels, temps, exit_layers, costs_by_model, taus, n_blocks, details=False):
    # Applies "exit at the first head whose calibrated normalised entropy < tau" OFFLINE to cached logits.
    # Identical to what forward_single_early_exit does per sample, but vectorised -> a whole tau sweep is cheap.
    exit_layers = sorted(exit_layers)
    N = labels.shape[0]
    ents = {l: normalized_entropy(logits[l], temps.get(l, 1.0)) for l in exit_layers}
    conf_exit = {l: F.softmax(logits[l] / temps.get(l, 1.0), -1).max(-1).values for l in exit_layers}
    pred_exit = {l: logits[l].argmax(-1) for l in exit_layers}
    pred_final = logits["final"].argmax(-1)
    conf_final = F.softmax(logits["final"] / temps.get("final", 1.0), -1).max(-1).values
    out = []
    for tau in taus:
        pred, conf = pred_final.clone(), conf_final.clone()
        used = torch.full((N,), n_blocks, dtype=torch.long)
        active = torch.ones(N, dtype=torch.bool)
        for l in exit_layers:
            m = active & (ents[l] < tau)
            pred[m], conf[m], used[m] = pred_exit[l][m], conf_exit[l][m], l
            active &= ~m
        frac = {str(l): float((used == l).float().mean()) for l in exit_layers}
        frac["final"] = float((used == n_blocks).float().mean())
        rec = {"tau": float(tau), "accuracy": float((pred == labels).float().mean()),
               "exit_fraction": frac, "mean_exit_layer": float(used.float().mean())}
        for name, cd in costs_by_model.items():
            per = torch.empty(N, dtype=torch.float64)
            per[used == n_blocks] = cd["final"]
            for l in exit_layers:
                per[used == l] = cd[l]
            rec[f"cost_{name}"] = float(per.mean())
        if details:
            rec.update(pred=pred.numpy(), conf=conf.numpy(), used=used.numpy())
        out.append(rec)
    return out

def pareto_front(points, cost_key):
    # maximise accuracy, minimise cost. Returned in ascending cost (and strictly ascending accuracy).
    front, best = [], -1.0
    for p in sorted(points, key=lambda p: (p[cost_key], -p["accuracy"])):
        if p["accuracy"] > best + 1e-12:
            front.append(p); best = p["accuracy"]
    return front

def knee_point(front, cost_key):
    # point of maximum distance above the chord of the normalised (cost, accuracy) curve
    if len(front) < 3:
        return front[0]
    c = np.array([p[cost_key] for p in front]); a = np.array([p["accuracy"] for p in front])
    if c.max() == c.min() or a.max() == a.min():
        return front[0]
    score = (a - a.min()) / (a.max() - a.min()) - (c - c.min()) / (c.max() - c.min())
    return front[int(score.argmax())]

def named_operating_points(points, cost_key, base_acc, eps):
    # conservative: cheapest point within 0.25*eps of baseline | knee: knee of the constraint-feasible front
    # aggressive: cheapest point within 2*eps (deliberately beyond the tolerance, to show what extra savings cost)
    feas = [p for p in points if p["accuracy"] >= base_acc - eps] or [max(points, key=lambda p: p["accuracy"])]
    wide = [p for p in points if p["accuracy"] >= base_acc - 2 * eps] or feas
    cons_pool = [p for p in points if p["accuracy"] >= base_acc - 0.25 * eps] or [max(points, key=lambda p: p["accuracy"])]
    return {"conservative": min(cons_pool, key=lambda p: p[cost_key]),
            "knee": knee_point(pareto_front(feas, cost_key), cost_key),
            "aggressive": pareto_front(wide, cost_key)[0]}

def marginal_table(points, cost_key):
    # walking from the most expensive front point to the cheapest: cost saved per accuracy POINT given up
    front = pareto_front(points, cost_key)[::-1]
    rows = []
    for prev, cur in zip(front[:-1], front[1:]):
        d_cost = prev[cost_key] - cur[cost_key]
        d_acc_pts = (prev["accuracy"] - cur["accuracy"]) * 100
        rows.append({"from_acc": prev["accuracy"], "to_acc": cur["accuracy"], "cost_saved": d_cost,
                     "acc_points_lost": d_acc_pts, "saved_per_acc_point": d_cost / max(d_acc_pts, 1e-9),
                     "to_cost": cur[cost_key], "to_key": cur.get("key"), "to_tau": cur["tau"]})
    # explicit columns: a one-point front gives no rows, and the CSV must still be readable
    return pd.DataFrame(rows, columns=["from_acc", "to_acc", "cost_saved", "acc_points_lost", "saved_per_acc_point", "to_cost", "to_key", "to_tau"])

def macro_f1(y_true, y_pred, n_classes=NUM_CLASSES):
    f1s = []
    for c in range(n_classes):
        tp = np.sum((y_pred == c) & (y_true == c)); fp = np.sum((y_pred == c) & (y_true != c)); fn = np.sum((y_pred != c) & (y_true == c))
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s))

def paired_bootstrap_diff(correct_a, correct_b, n=2000, seed=0):
    d = np.asarray(correct_a, float) - np.asarray(correct_b, float)
    rng = np.random.RandomState(seed)
    means = np.array([d[rng.randint(0, len(d), len(d))].mean() for _ in range(n)])
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


## 1. Load everything

In [ ]:
# ---------------- load test data, backbone, cost models, operating points, checkpoints ----------------
Xtr, Ytr, Xte, Yte = load_cifar_tensors()
test_data = GPUData(Xte, Yte)
test_idx = list(range(len(Yte)))
backbone, geom = make_backbone()
meter = EnergyMeter()
UNIT, U = ("mJ", 1000.0) if meter.unit == "J" else ("ms-equivalent (wall-clock s x1000)", 1000.0)

ops = json.load(open(f"{RESULTS_DIR}/operating_points.json"))
cost_table = json.load(open(f"{RESULTS_DIR}/cost_model_energy.json"))
cost_models = {"energy": MeasuredCostModel(cost_table), "macs": MACCostModel(geom), "tokens": TokenCostModel()}
def costs_for(cfg):
    return {name: path_costs(cfg, cm, geom) for name, cm in cost_models.items()}

models, cfgs, test_logits = {}, {}, {}
for h in sorted({o["hash"] for o in ops.values()}):
    models[h], cfgs[h], _ = load_ckpt(f"{CHECKPOINT_DIR}/model_{h}.pt", backbone)
    test_logits[h], test_labels = collect_logits(models[h], test_data, test_idx)
    print("loaded", h, cfg_label(cfgs[h]))
print("operating points:", list(ops))


## 2. Test-set metrics

In [ ]:
# ---------------- test-set metrics at each operating point (tau fixed on validation, never tuned on test) ----------------
rows, det = [], {}
y_true = test_labels.numpy()
for name, op in ops.items():
    h, cfg, m = op["hash"], cfgs[op["hash"]], models[op["hash"]]
    d = sweep_points(test_logits[h], test_labels, m.temps, cfg["exit_layers"], costs_for(cfg), [op["tau"]], geom["blocks"], details=True)[0]
    det[name] = d
    correct = (d["pred"] == y_true).astype(int)
    rows.append({"name": name, "pipeline": op["pipeline"], "config": cfg_label(cfg), "tau": op["tau"],
                 "test_acc": d["accuracy"], "f1_macro": macro_f1(y_true, d["pred"]),
                 "ece": expected_calibration_error(d["conf"], correct), "mean_exit_layer": d["mean_exit_layer"],
                 "pred_energy": d["cost_energy"] * U, "macs_M": d["cost_macs"] / 1e6, "token_layers": d["cost_tokens"],
                 "exit_fraction": json.dumps({k: round(v, 3) for k, v in d["exit_fraction"].items()})})
base_correct = (det["B0"]["pred"] == y_true).astype(int)
for r in rows:
    c = (det[r["name"]]["pred"] == y_true).astype(int)
    dm, lo, hi = paired_bootstrap_diff(c, base_correct)
    r.update(acc_diff_vs_B0=dm, acc_diff_ci_lo=lo, acc_diff_ci_hi=hi)
acc_df = pd.DataFrame(rows)
acc_df.round(4)


## 3. Real energy and latency

In [ ]:
# ---------------- REAL energy / latency: single-sample, genuine early exit, interleaved repeats ----------------
sel = stratified_indices(Yte.numpy(), max(1, REAL_ENERGY_SAMPLES // NUM_CLASSES), SEED)
xs = torch.cat([prepare_batch(test_data.X[torch.as_tensor(sel[i:i + 100], device=device)]) for i in range(0, len(sel), 100)])
print("real-energy sample count:", xs.shape[0])

def measure_once(model, tau):
    lat = []
    with torch.no_grad(), amp_ctx():
        for i in range(min(20, xs.shape[0])):                        # warm-up (not measured)
            model.forward_single_early_exit(xs[i:i + 1], tau)
        e0 = meter.read()
        for i in range(xs.shape[0]):
            t0 = time.perf_counter()
            model.forward_single_early_exit(xs[i:i + 1], tau)
            if device == "cuda":
                torch.cuda.synchronize()
            lat.append((time.perf_counter() - t0) * 1000)
        e1 = meter.read()
    return {"energy_per_sample": (e1 - e0) / xs.shape[0], "lat_mean_ms": float(np.mean(lat)), "lat_p95_ms": float(np.percentile(lat, 95))}

rng_e = np.random.RandomState(SEED)
raw = []
for rep in range(N_ENERGY_REPEATS):
    order = list(ops); rng_e.shuffle(order)                          # interleave configs so drift hits all of them
    for name in order:
        op = ops[name]
        r = measure_once(models[op["hash"]], op["tau"]); r.update(name=name, rep=rep); raw.append(r)
        print(f"rep {rep} {name:<7} {r['energy_per_sample'] * U:9.3f} {UNIT}/sample  lat {r['lat_mean_ms']:.2f} ms")
raw_df = pd.DataFrame(raw)
raw_df.to_csv(f"{RESULTS_DIR}/real_energy_raw.csv", index=False)
agg = raw_df.groupby("name").agg(meas_energy=("energy_per_sample", "mean"), meas_energy_std=("energy_per_sample", "std"),
                                 lat_mean_ms=("lat_mean_ms", "mean"), lat_p95_ms=("lat_p95_ms", "mean")).reset_index()
agg[["meas_energy", "meas_energy_std"]] *= U
agg


## 4. Comparison table

In [ ]:
# ---------------- final comparison table ----------------
df = acc_df.merge(agg, on="name")
b0 = df[df.name == "B0"].iloc[0]
df["macs_rel_B0"] = df["macs_M"] / b0["macs_M"]
df["pred_energy_rel_B0"] = df["pred_energy"] / b0["pred_energy"]
df["meas_energy_rel_B0"] = df["meas_energy"] / b0["meas_energy"]
if meter.unit == "J":
    kwh = df["meas_energy"] / U / 3.6e6
    df["co2_g_per_1k_inferences"] = kwh * CARBON_INTENSITY_KG_PER_KWH * 1000 * 1000
df["acc_per_mJ"] = df["test_acc"] / df["meas_energy"]
df.to_csv(f"{RESULTS_DIR}/full_comparison.csv", index=False)
show = ["name", "config", "tau", "test_acc", "acc_diff_vs_B0", "ece", "mean_exit_layer", "macs_rel_B0", "pred_energy_rel_B0", "meas_energy_rel_B0", "meas_energy", "meas_energy_std", "lat_mean_ms"]
print(f"energy unit: {UNIT} per sample (" + ("measured = GPU-0 NVML counter, includes idle draw during the window" if meter.ok else "NVML unavailable: wall-clock fallback") + ")")
df[show].round(4)


## 5. Curves and plots

In [ ]:
# ---------------- tau curves on test (for plots only; operating points were fixed on validation) ----------------
curves = {}
for h, cfg in cfgs.items():
    curves[h] = pd.DataFrame(sweep_points(test_logits[h], test_labels, models[h].temps, cfg["exit_layers"], costs_for(cfg), TAU_GRID, geom["blocks"]))
    curves[h]["energy"] = curves[h]["cost_energy"] * U
    curves[h]["label"] = cfg_label(cfg)


In [ ]:
# ---------------- plots ----------------
colors = {"baseline": "black", "fixed-exit": "gray", "energy": "tab:green", "mac": "tab:red"}

# 1) accuracy vs measured energy
fig, ax = plt.subplots(figsize=(7, 5))
for _, r in df.iterrows():
    ax.errorbar(r["meas_energy"], r["test_acc"], xerr=r["meas_energy_std"], fmt="o", color=colors[r["pipeline"]], capsize=3)
    ax.annotate(r["name"], (r["meas_energy"], r["test_acc"]), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.axhline(b0["test_acc"] - EPS, ls="--", color="gray", lw=0.8, label=f"B0 - {EPS:.0%} tolerance")
ax.set_xlabel(f"measured energy per sample ({UNIT})"); ax.set_ylabel("test accuracy"); ax.legend(); ax.set_title("Accuracy vs measured energy")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_accuracy_vs_energy.png", dpi=150); plt.show()

# 2) MAC reduction vs measured-energy reduction
fig, ax = plt.subplots(figsize=(8, 4.5))
names = [n for n in df.name if n != "B0"]; sub = df[df.name.isin(names)]
x = np.arange(len(names)); w = 0.27
ax.bar(x - w, 100 * (1 - sub["macs_rel_B0"]), w, label="MACs saved (%)")
ax.bar(x, 100 * (1 - sub["pred_energy_rel_B0"]), w, label="energy saved, cost-model prediction (%)")
ax.bar(x + w, 100 * (1 - sub["meas_energy_rel_B0"]), w, label="energy saved, measured (%)")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=45); ax.set_ylabel("saving vs B0 (%)"); ax.legend(fontsize=8); ax.set_title("MAC savings vs real energy savings")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_macs_vs_energy.png", dpi=150); plt.show()

# 3) predicted vs measured energy
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df["pred_energy"], df["meas_energy"])
for _, r in df.iterrows():
    ax.annotate(r["name"], (r["pred_energy"], r["meas_energy"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
lim = [0, max(df["pred_energy"].max(), df["meas_energy"].max()) * 1.1]; ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlabel(f"cost-model prediction ({UNIT})"); ax.set_ylabel(f"measured ({UNIT})"); ax.set_title("Cost-model validation")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_pred_vs_measured.png", dpi=150); plt.show()

# 4) tau curves per config (test), operating points marked
fig, ax = plt.subplots(figsize=(8, 5))
for h, c in curves.items():
    ax.plot(c["energy"], c["accuracy"], marker=".", lw=1, label=c["label"].iloc[0][:38])
for _, r in df.iterrows():
    ax.scatter(r["pred_energy"], r["test_acc"], s=60, edgecolor="k", facecolor="none")
ax.set_xlabel(f"predicted energy ({UNIT})"); ax.set_ylabel("test accuracy"); ax.legend(fontsize=6); ax.set_title("Accuracy-energy curves over the entropy threshold")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_tau_curves.png", dpi=150); plt.show()

# 5) marginal efficiency (validation front, energy-driven): energy saved per accuracy point lost
mt = pd.read_csv(f"{RESULTS_DIR}/marginal_energy_val.csv")
if len(mt):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(range(len(mt)), mt["saved_per_acc_point"] * U)
    ax.set_xticks(range(len(mt))); ax.set_xticklabels([f"{a:.3f}\u2192{b:.3f}" for a, b in zip(mt.from_acc, mt.to_acc)], rotation=60, fontsize=7)
    ax.set_ylabel(f"{UNIT} saved per accuracy point lost"); ax.set_xlabel("step along the Pareto front (accuracy)"); ax.set_title("Marginal efficiency along the energy-driven front (validation)")
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_marginal_efficiency.png", dpi=150); plt.show()

# 6) calibration: ECE before / after (validation)
calib = json.load(open(f"{RESULTS_DIR}/calibration_results.json"))
labels_, before, after = [], [], []
for h, rep in calib.items():
    for k in rep["ece_before"]:
        labels_.append(f"{h[:4]}:{k}"); before.append(rep["ece_before"][k]); after.append(rep["ece_after"][k])
fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(len(labels_))
ax.bar(x - 0.2, before, 0.4, label="before"); ax.bar(x + 0.2, after, 0.4, label="after temperature scaling")
ax.set_xticks(x); ax.set_xticklabels(labels_, rotation=60, fontsize=7); ax.set_ylabel("ECE"); ax.legend(); ax.set_title("Calibration per head (validation split)")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_calibration.png", dpi=150); plt.show()

# 7) exit distribution per operating point
fig, ax = plt.subplots(figsize=(8, 4.5)); bottom = np.zeros(len(df))
keys = sorted({k for r in df["exit_fraction"] for k in json.loads(r)}, key=lambda k: (k == "final", int(k) if k != "final" else 99))
for k in keys:
    vals = np.array([json.loads(r).get(k, 0.0) for r in df["exit_fraction"]])
    ax.bar(df["name"], vals, bottom=bottom, label=f"exit {k}"); bottom += vals
ax.set_ylabel("fraction of test samples"); ax.legend(fontsize=7); ax.set_title("Where samples exit"); plt.xticks(rotation=45)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/plot_exit_distribution.png", dpi=150); plt.show()


## 6. No-regression check

In [ ]:
# ---------------- no-regression check (paired bootstrap, 95% CI) ----------------
print(f"No-regression rule: test accuracy >= B0 - {EPS:.0%}   (B0 test acc = {b0['test_acc']:.4f})\n")
for _, r in df.iterrows():
    if r["name"] == "B0":
        continue
    ok = r["acc_diff_vs_B0"] >= -EPS
    expected = " (aggressive points exceed the tolerance by design)" if r["name"].endswith("aggr") and not ok else ""
    print(f"  {'OK ' if ok else 'FAIL'} {r['name']:<7} diff={r['acc_diff_vs_B0']:+.4f}  95% CI [{r['acc_diff_ci_lo']:+.4f}, {r['acc_diff_ci_hi']:+.4f}]  "
          f"energy vs B0 (measured) = {r['meas_energy_rel_B0']:.2f}x{expected}")


In [ ]:
zip_path = os.path.join(WORKING_DIR, "early_exit_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [CHECKPOINT_DIR, RESULTS_DIR]:
        for f in sorted(os.listdir(folder)):
            z.write(os.path.join(folder, f), arcname=os.path.join(os.path.basename(folder), f))
print("zip refreshed:", zip_path)
print(sorted(os.listdir(RESULTS_DIR)))
